### Initialization and Environment Setup

This cell initializes the AutoREACTER environment and prepares the working directory for the current run.

In [ ]:
from AutoREACTER.initialization import Initialization
from AutoREACTER.input_parser import InputParser
from AutoREACTER.cache import GetCacheDir, RunDirectoryManager
from AutoREACTER.detectors.functional_groups_detector import FunctionalGroupsDetector
from AutoREACTER.detectors.reaction_detector import ReactionDetector
from AutoREACTER.detectors.non_monomer_detector import NonReactantsDetector
from AutoREACTER.reaction_preparation.reaction_processor.prepare_reactions import PrepareReactions
from AutoREACTER.reaction_preparation.lunar_client.molecule_3d_preparation import Molecule3DPreparation
from AutoREACTER.reaction_preparation.lunar_client.lunar_api_wrapper import LunarAPIWrapper
from AutoREACTER.reaction_preparation.lunar_client.REACTER_files_builder import REACTERFilesBuilder

import json


Initialization()

input_parser = InputParser()

cache_manager = GetCacheDir()
cache_dir = cache_manager.staging_dir

run_manager = RunDirectoryManager(cache_manager.cache_base_dir)
output_dir = run_manager.make_dated_run_dir()

print(f"[INFO] Staging directory: {cache_dir}")
print(f"[INFO] Final output directory: {output_dir}")

### Cache Cleanup (Optional)

This cell can be used to clean up cached data generated by previous runs.


In [ ]:
# Clean up cached data from the AutoREACTER cache directory.
# Useful for freeing disk space or resetting the cache state.

# from AutoREACTER.cache import RetentionCleanup
# RetentionCleanup.run(GetCacheDir().cache_base_dir)

### Load and Validate Monomers

This cell loads the example input file, validates it using the `InputParser`, and visualizes the monomers.



In [ ]:
# Load input data and validate structure.
# The validation step normalizes inputs and prepares molecular representations.

with open("example_1_inputs_count_mode.json", "r") as f:
    input_data = json.load(f)

validated_inputs = input_parser.validate_inputs(input_data)


### Visualize Monomers (Optional)

In [ ]:
# Generate and display the initial monomer structures.

initial_molecules = input_parser.initial_molecules_image_grid(validated_inputs)
initial_molecules

### Functional Group Detection

This cell runs the functional group detection step.


In [ ]:
# Detect functional groups to get valid monomers

functional_groups_detector = FunctionalGroupsDetector()

functional_groups, functional_groups_imgs = (
    functional_groups_detector.functional_groups_detector(
        validated_inputs.monomers
    )
)

### Functional Group Visualization (Optional)

This cell visualizes the detected functional groups on the molecules.

In [ ]:
# The resulting image shows each molecule with the detected functional groups marked.

img = functional_groups_detector.functional_group_highlighted_molecules_image_grid(functional_groups_imgs)
img

### Reaction Detection

This cell identifies possible reactions between the detected functional groups.


In [ ]:
# Detect possible reactions based on identified functional groups.

reaction_detector = ReactionDetector()
reaction_instances = reaction_detector.reaction_detector(functional_groups)

### Reaction visualization (Optional)

In [ ]:
# Visualize detected reactions.

img = reaction_detector.available_reaction_image_grid(reaction_instances)
img

### Reaction Selection

This cell filters and selects the relevant reactions from the detected reaction instances.


In [ ]:
# Select relevant reactions for template generation as user intened.

selected_reactions = reaction_detector.reaction_selection(reaction_instances)

### Non-Reactant (Non-Monomer) Detection

This cell identifies molecules that do **not participate in any detected reactions**.



In [ ]:
# Identify non-reactive molecules based on the selected reactions.

non_monomer_detector = NonReactantsDetector()
non_reactants_list = non_monomer_detector.non_monomer_detector(
    validated_inputs,
    selected_reactions
)


### Non reactants molecules visualization

In [ ]:
# Visualize non-reactive molecules (if any are detected).

img_non_reactants = non_monomer_detector.non_reactants_to_visualization(non_reactants_list)
img_non_reactants

### Update Inputs After Non-Reactant Filtering

This cell updates the validated inputs after identifying non-reactive molecules.


In [ ]:
# Update inputs by filtering out non-reactive molecules.

updated_inputs = non_monomer_detector.non_reactant_selection(
    validated_inputs,
    non_reactants_list
)

### Prepare reaction templates from the selected reactions for downstream processing.

In [ ]:
prepare_reactions = PrepareReactions(cache_dir)
prepared_reactions = prepare_reactions.prepare_reactions(selected_reactions)

### Reaction Template Visualization

Visualize reaction templates with different highlighting options.

- **Default** or **template**: Highlights all structural changes in the reaction templates  
- **edge**: Highlights edge atoms of the templates  
- **delete**: Highlights removed components (if applicable)  
- **initiators**: Highlights reaction initiator atoms  

In [ ]:
img = prepare_reactions.reaction_templates_highlighted_image_grid(prepared_reactions)
img

### 3D Geometry Preparation

Prepare 3D molecular geometries for both inputs molecules and reaction templates.  
This step generates the structures required for the Lunar atom typing.

In [ ]:
molecule3dpreparation = Molecule3DPreparation(cache_dir)

updated_inputs_with_3d_mols, prepared_reactions_with_3d_mols = (
    molecule3dpreparation.prepare_molecule_3d_geometry(
        updated_inputs,
        prepared_reactions
    )
)  

### Lunar Workflow

Run the Lunar workflow to generate simulation-ready files from prepared 3D structures.

In [ ]:
lunar_api_wrapper = LunarAPIWrapper(cache_dir)

lunar_results = lunar_api_wrapper.lunar_workflow(
    updated_inputs_with_3d_mols,
    prepared_reactions_with_3d_mols
)

### REACTER File Generation and Finalization

Generate REACTER-compatible files from the Lunar results and prepared reactions.  
The generated files are then moved to the final run directory, and all internal paths are updated accordingly.

In [ ]:
REACTER_files_builder = REACTERFilesBuilder(
    cache_dir=cache_dir,
    updated_inputs_with_3d_mols=updated_inputs_with_3d_mols
)

reacter_files = REACTER_files_builder.molecule_template_preparation(
    lunar_files=lunar_results,
    prepared_reactions_with_3d_mols=prepared_reactions_with_3d_mols
)

reacter_files = run_manager.move_reacter_files(
    reacter_files,
    staging_dir=cache_dir,
    final_dir=output_dir
)